# Notebook 5: Modeling & Prediction

## Purpose
Model whether theory-aligned categories & indices predict:
1. **Goodreads rating**
2. **Popularity tier (Top vs Trash)**

## Models
- **Logistic regression** (Top vs Trash)
- **OLS regression** (`avg_rating`)
- **Key interactions** (e.g. `Luxury × Love`, `Protective–Jealousy`)
- **Controls**: `length`, `author_id`, `year` (if available)

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150
np.random.seed(42)

## 1. Load Data

In [ ]:
PROJECT_ROOT = Path().resolve().parent.parent.parent.parent
INPUT_FILE = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis" / "indices_book.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis"

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} books")
print(f"Columns: {list(df.columns)}")

## 2. Prepare Features and Targets

In [ ]:
# Define index features
index_features = ['love_over_sex', 'hea_index', 'luxury_x_love', 
                  'protective_minus_jealous', 'dark_vs_tender', 'miscommunication_balance']
available_indices = [idx for idx in index_features if idx in df.columns]

# Control variables
control_vars = []
if 'length_tokens' in df.columns:
    control_vars.append('length_tokens')
elif 'length_words' in df.columns:
    control_vars.append('length_words')
if 'year' in df.columns:
    control_vars.append('year')

# Prepare feature matrix
feature_cols = available_indices + control_vars
X = df[feature_cols].fillna(0)

# Targets
y_rating = df['average_rating_weighted_mean'] if 'average_rating_weighted_mean' in df.columns else None
y_group = (df['group'] == 'Top').astype(int) if 'group' in df.columns else None

print(f"Features: {feature_cols}")
print(f"Sample size: {len(X)}")

## 3. Logistic Regression: Top vs Trash

In [ ]:
if y_group is not None:
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_group, test_size=0.2, random_state=42, stratify=y_group
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Fit logistic regression
    log_reg = LogisticRegression(random_state=42, max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = log_reg.predict(X_test_scaled)
    y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]
    
    # Evaluation
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nLogistic Regression Accuracy: {accuracy:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Trash', 'Top']))
    
    # Feature importance (coefficients)
    coef_df = pd.DataFrame({
        'feature': feature_cols,
        'coefficient': log_reg.coef_[0],
        'abs_coefficient': np.abs(log_reg.coef_[0])
    }).sort_values('abs_coefficient', ascending=False)
    
    print("\nFeature Coefficients:")
    print(coef_df)
    
    # Plot coefficients
    plt.figure(figsize=(10, 6))
    sns.barplot(data=coef_df, x='coefficient', y='feature', orient='h')
    plt.title('Logistic Regression Coefficients (Top vs Trash)')
    plt.xlabel('Coefficient')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'logistic_regression_coefficients.png')
    plt.show()

## 4. OLS Regression: Predicting Rating

In [ ]:
if y_rating is not None:
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_rating, test_size=0.2, random_state=42
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Fit OLS regression
    ols_reg = LinearRegression()
    ols_reg.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = ols_reg.predict(X_test_scaled)
    
    # Evaluation
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"\nOLS Regression R²: {r2:.3f}")
    print(f"RMSE: {rmse:.3f}")
    
    # Feature importance (coefficients)
    coef_df = pd.DataFrame({
        'feature': feature_cols,
        'coefficient': ols_reg.coef_,
        'abs_coefficient': np.abs(ols_reg.coef_)
    }).sort_values('abs_coefficient', ascending=False)
    
    print("\nFeature Coefficients:")
    print(coef_df)
    
    # Plot coefficients with confidence intervals (using statsmodels)
    X_with_const = sm.add_constant(X_train_scaled)
    model = sm.OLS(y_train, X_with_const).fit()
    
    print("\nStatsmodels OLS Summary:")
    print(model.summary())
    
    # Plot predicted vs actual
    plt.figure(figsize=(8, 8))
    plt.scatter(y_test, y_pred, alpha=0.6)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    plt.xlabel('Actual Rating')
    plt.ylabel('Predicted Rating')
    plt.title(f'OLS Regression: Predicted vs Actual Rating (R²={r2:.3f})')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'ols_regression_predicted_vs_actual.png')
    plt.show()

## 5. Interaction Effects

In [ ]:
# Create interaction terms
if 'luxury_x_love' in df.columns and 'protective_minus_jealous' in df.columns:
    # Example: Luxury × Love interaction with Protective–Jealousy
    X_interaction = X.copy()
    X_interaction['luxury_love_x_protective_jealous'] = (
        X_interaction.get('luxury_x_love', 0) * 
        X_interaction.get('protective_minus_jealous', 0)
    )
    
    # TODO: Fit models with interaction terms
    # Compare model fit with and without interactions

## 6. Save Results

In [ ]:
# Save model coefficients and performance metrics
if 'coef_df' in locals():
    output_file = OUTPUT_DIR / "model_coefficients.csv"
    coef_df.to_csv(output_file, index=False)
    print(f"✓ Saved: {output_file}")

## Summary

Modeling complete. Next: Notebook 6 (Timecourse Analysis)